# PRISMA Nanonis Test

In [ ]:
%load_ext line_profiler
%load_ext autoreload
%autoreload 2

In [ ]:
from functools import partial
import matplotlib.pyplot as plt
import nanonis_spm
import numpy as np
from os.path import join
from scipy.optimize import curve_fit
from scipy.stats import norm
import socket

from prisma.apf.api import ActiveParameterFitRP
from prisma.experiment import SpectroscopySeriesRP
from prisma.physics_models.jax import beem_integral_f_vector_j
from prisma.physics_models.python import Ibeem_vector
from prisma.plot.plot_lib import plot_function_targets
from prisma.samplers.instruments.nanonis import NanonisSamplerSpectroscopyTap
from prisma.samplers.pseudorandom.noise import RandomNoiseSampler
from prisma.samplers.pseudorandom.function import \
    PrecomputedNoiseVectorizedJAXFunctionDomainSampler

In [ ]:
np.random.seed(0)

import jax
jax.config.update('jax_enable_x64', True)

In [ ]:
save_dir = 'BEEM_Data\\Nanonis_BNLR_Test_7'

# Nanonis API Testing

In [ ]:
connection = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
connection.connect(("localhost",6502))
nanonisInstance = nanonis_spm.Nanonis(connection)
nanonisInstance.returnDebugInfo(1)

In [ ]:
v0test = 0.5
vftest = 1.2
vrange = np.arange(v0test, vftest, 10)

In [ ]:
test_acq = []
for v in vrange:
    nanonisInstance.Bias_Set(v)
    ch_read = nanonisInstance.Signals_ValGet(1, True)[2]
    test_acq.append(ch_read)

In [ ]:
plt.plot(test_acq)

In [ ]:
nanonisInstance.BiasSpectr_LimitsSet(np.array(v0test, dtype=np.float32),
                                     np.array(vftest, dtype=np.float32))

In [ ]:
Spectr_AcquiredData = nanonisInstance.BiasSpectr_Start(1, "Example_BiasSpectroscopy")[2][5]
AcqSignal_I_BEEM = Spectr_AcquiredData[1]
AcqSignal_Bias = Spectr_AcquiredData[2]

In [ ]:
plt.plot(AcqSignal_Bias, AcqSignal_I_BEEM)

In [ ]:
nanonisInstance.close()

# Parameter Simulation

## Sample Simulation

In [ ]:
N = 100
x0 = 1.25
xf = 0.4
x = np.linspace(x0, xf, N) # (N x 1)
w = np.array([-0.03, .8, -1e-5]) # (M x 1)
nonlin_f_w = partial(beem_integral_f_vector_j, w=w)
sigma_f_sim = .001

In [ ]:
t_noise_rv = norm(scale=sigma_f_sim)
t_noise_sampler = RandomNoiseSampler(t_noise_rv)
t_noise_np = np.array(t_noise_sampler.sample(N))
t_noise_np = np.array(t_noise_sampler.sample(N))
t_sampler = PrecomputedNoiseVectorizedJAXFunctionDomainSampler(nonlin_f_w, t_noise_np)

In [ ]:
y = np.array(nonlin_f_w(x))
t = np.array([t_sampler.sample(x_i) for x_i in x])

In [ ]:
plot_function_targets(x, y, t, function_label="function $y$")

## Scale Testing

In [ ]:
p0_cf = np.array([0., .8, -1e-5])
nonlin_f_cf = Ibeem_vector

In [ ]:
cf_opt, cf_cov = curve_fit(nonlin_f_cf, x, t, p0=p0_cf)

In [ ]:
cf_opt

In [ ]:
y_cf = nonlin_f_cf(x, *cf_opt)

In [ ]:
plot_function_targets(x, y_cf, t, function_label="fit")

In [ ]:
scale_coef = -1.
in_min = 0.
in_max = .2
out_min = 0.
out_max = 100.

In [ ]:
t_sampler.reset()
t_sampler.scale_coef = scale_coef
t_sampler.set_rescaling_bounds(in_min, in_max, out_min, out_max)
t_p = np.array([t_sampler.sample(x_i) for x_i in x])

In [ ]:
cf_opt_p, cf_cov_p = curve_fit(nonlin_f_cf, x, t_p,)

In [ ]:
cf_opt_p

In [ ]:
y_cf_p = nonlin_f_cf(x, *cf_opt_p)

In [ ]:
plot_function_targets(x, y_cf_p, t_p, function_label="fit")

# Nanonis Sampler

In [ ]:
nanonis_sampler = NanonisSamplerSpectroscopyTap()
nanonis_sampler.scale_coef = scale_coef
nanonis_sampler.set_rescaling_bounds(in_min, in_max, out_min, out_max)
nanonis_sampler.set_store_unprocessed(join(save_dir,
                                           f"uproc_meas_test.npy"))

# Active Parameter Fit Test Run

In [ ]:
sample_ct = 33
m_0 = np.array([15., .8, 10e-3])
sigma_0 = np.diag([10., .5, 5e-3])
S_0 = np.power(sigma_0, 2.)
sigma_f = 2. # function standard deviation
var_f = sigma_f**2 # function variance
beta = (1. / var_f) # function precision
magnitude_rp = 1e-2
scale_rp = 1e-2
nonlin_f = beem_integral_f_vector_j

In [ ]:
apf = ActiveParameterFitRP(beem_integral_f_vector_j, x, nanonis_sampler, beta, m_0, S_0, sample_ct, magnitude_rp, scale_rp)

In [ ]:
apf.fit()

In [ ]:
apf.init_np_arrs()

In [ ]:
apf.save_np_arrs(save_dir)
nanonis_sampler.save_unprocessed()

In [ ]:
apf.m_prior_np[-1]

In [ ]:
apf.var_prior_np[-1]

In [ ]:
len(nanonis_sampler.unprocessed_measurements)

In [ ]:
apf.plot_predictive_acq_train_iter(y=y_cf_p)

In [ ]:
apf.plot_w_rmse(cf_opt_p)

In [ ]:
apf.plot_w_i_err(cf_opt_p, 1, w_i_name="$X_0$")

In [ ]:
apf.plot_predictive_rmse(t_p, y_cf_p)

In [ ]:
%timeit apf.fit()

13.5 ms ± 180 μs per loop (mean ± std. dev. of 7 runs, 100 loops each)


In [ ]:
%lprun -f apf.fit apf.fit()

Timer unit: 1e-07 s

Total time: 0.0154206 s
File: C:\Users\djbasil\Documents\GitHub\q-sense-optim\src\prisma\apf\api\active_parameter_fit.py
Function: AbstractActiveParameterFit.fit at line 37

Line #      Hits         Time  Per Hit   % Time  Line Contents
    37                                               def fit(self):
    38        11        170.0     15.5      0.1          for i in range(self.sample_ct):
    39        10      12280.0   1228.0      8.0              x_next, acq_val_next = self._acquisition_f(i)
    40        10     127960.0  12796.0     83.0              t_next = self.t_sampler.sample(x_next)
    41        10        786.0     78.6      0.5              self.x_n_jnp, self.t_n_jnp, self.acq_val_jnp, \
    42        10        731.0     73.1      0.5                  self.phi_n_jnp, self.m_prior_jnp, self.var_prior_jnp \
    43        20      12051.0    602.5      7.8                  = posterior_update(x_next, acq_val_next, t_next,
    44        10         68.0      

In [ ]:
%lprun -f NanonisSamplerSpectroscopyTap.sample apf.fit()

# Active Parameter Fit Spectroscopy Series

In [ ]:
series_ct = 100
nanonis_sampler.set_store_unprocessed(join(save_dir,
                                           f"uproc_meas_spectr_series.npy"))

In [ ]:
exp = SpectroscopySeriesRP(nanonis_sampler, series_ct,
                           nonlin_f, x, beta,
                           m_0, S_0, sample_ct,
                           magnitude_rp, scale_rp)

In [ ]:
exp.run()

In [ ]:
exp.init_spectroscopy_series_np()

In [ ]:
exp.save_spectroscopy_series(save_dir)

In [ ]:
nanonis_sampler.save_unprocessed()

In [ ]:
exp.m_prior_series_np.shape

In [ ]:
exp.m_prior_series_np[:, -1, 1]

In [ ]:
np.mean(exp.m_prior_series_np[:, -1, 1])

In [ ]:
np.std(exp.m_prior_series_np[:, -1, 1])

In [ ]:
np.mean(np.sqrt(exp.var_prior_series_np[:, -1, 1, 1]))

In [ ]:
np.std(np.sqrt(exp.var_prior_series_np[:, -1, 1, 1]))

In [ ]:
len(nanonis_sampler.unprocessed_measurements)

In [ ]:
nanonis_sampler.close()